In [62]:
import json

PATH = "/kaggle/input/datasets/akhilkumar060/candidates-jsonl/candidates.jsonl"

with open(PATH, "r", encoding="utf-8") as f:
    first_candidate = json.loads(next(f))

print(first_candidate.keys())

dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [63]:
from pprint import pprint

pprint(first_candidate["profile"])
pprint(first_candidate["career_history"][:2])
pprint(first_candidate["skills"][:10])
pprint(first_candidate["redrob_signals"])

{'anonymized_name': 'Ira Vora',
 'country': 'Canada',
 'current_company': 'Mindtree',
 'current_company_size': '10001+',
 'current_industry': 'IT Services',
 'current_title': 'Backend Engineer',
 'headline': 'Backend Engineer | SQL, Spark, Cloud',
 'location': 'Toronto',
 'summary': 'Software / data professional with 6.9 years of experience '
            'building data pipelines, backend systems, and analytics '
            "infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL "
            "warehouses are home territory; I'm building competence on the ML "
            'side. My toolkit is solid on the data engineering side — Python, '
            "SQL, Spark, Airflow, warehouse design — and I've completed a "
            'couple of self-directed ML projects (Kaggle competitions, side '
            'projects fine-tuning small models). Interested in transitioning '
            'toward more AI/ML-focused work, ideally at a company where I can '
            'leverage my existin

In [64]:
count = 0

with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        count += 1

print(count)

100000


In [65]:
import json

title_count = {}

with open(PATH, "r", encoding="utf-8") as f:
    for line in f:

        c = json.loads(line)

        title = c["profile"]["current_title"]

        title_count[title] = title_count.get(title, 0) + 1

print(sorted(title_count.items(), key=lambda x: x[1], reverse=True)[:50])

[('Business Analyst', 5833), ('HR Manager', 5830), ('Mechanical Engineer', 5791), ('Accountant', 5764), ('Project Manager', 5754), ('Customer Support', 5750), ('Operations Manager', 5744), ('Content Writer', 5727), ('Sales Executive', 5713), ('Civil Engineer', 5702), ('Graphic Designer', 5689), ('Marketing Manager', 5524), ('Software Engineer', 3450), ('Full Stack Developer', 2873), ('Cloud Engineer', 2836), ('Java Developer', 2809), ('.NET Developer', 2788), ('DevOps Engineer', 2787), ('Mobile Developer', 2757), ('Frontend Engineer', 2738), ('QA Engineer', 2682), ('Analytics Engineer', 764), ('Data Engineer', 744), ('Data Analyst', 728), ('Backend Engineer', 704), ('Senior Data Engineer', 687), ('Senior Software Engineer', 653), ('ML Engineer', 167), ('AI Research Engineer', 153), ('Data Scientist', 145), ('Senior Software Engineer (ML)', 142), ('Computer Vision Engineer', 132), ('Junior ML Engineer', 131), ('AI Specialist', 130), ('Recommendation Systems Engineer', 26), ('Machine Lea

In [66]:
from collections import Counter

skill_counter = Counter()

with open(PATH, "r", encoding="utf-8") as f:
    for line in f:

        c = json.loads(line)

        title = c["profile"]["current_title"].lower()

        if any(x in title for x in [
            "ai",
            "ml",
            "machine learning",
            "data scientist",
            "nlp"
        ]):
            for skill in c["skills"]:
                skill_counter[skill["name"]] += 1

print(skill_counter.most_common(100))

[('Feature Engineering', 278), ('MLflow', 278), ('Diffusion Models', 274), ('Image Classification', 273), ('MLOps', 273), ('Time Series', 271), ('Speech Recognition', 269), ('OpenCV', 264), ('Object Detection', 260), ('YOLO', 260), ('GANs', 259), ('ASR', 258), ('Reinforcement Learning', 256), ('Data Science', 256), ('BentoML', 255), ('TTS', 255), ('Weights & Biases', 255), ('Kubeflow', 254), ('Forecasting', 254), ('Computer Vision', 254), ('Statistical Modeling', 250), ('CNN', 237), ('Python', 213), ('PyTorch', 213), ('Qdrant', 209), ('NLP', 209), ('Hugging Face Transformers', 206), ('Semantic Search', 206), ('BM25', 206), ('OpenSearch', 203), ('QLoRA', 203), ('Deep Learning', 201), ('Weaviate', 200), ('LoRA', 199), ('Milvus', 197), ('pgvector', 195), ('Pinecone', 194), ('LangChain', 193), ('Elasticsearch', 191), ('PEFT', 186), ('Information Retrieval', 185), ('Learning to Rank', 185), ('scikit-learn', 184), ('Embeddings', 184), ('Machine Learning', 183), ('Prompt Engineering', 181), (

In [67]:
TITLE_WEIGHTS = {
    "recommendation systems engineer": 30,
    "search engineer": 30,
    "senior machine learning engineer": 28,
    "staff machine learning engineer": 28
}

In [68]:
GOLD_SKILLS = {
    "information retrieval": 12,
    "learning to rank": 12,
    "embeddings": 12,
    "vector search": 12,
    "rag": 12
}

In [69]:
CAREER_KEYWORDS = [
    "ranking",
    "retrieval",
    "recommendation",
    "search",
    "embedding",
    "vector",
    "semantic",
    "rag",
    "llm",
    "ndcg",
    "mrr"
]

In [70]:
def score_candidate(c):

    score = 0

    profile = c["profile"]

    exp = profile["years_of_experience"]

    # Experience Score
    if 5 <= exp <= 9:
        score += 20
    elif 4 <= exp <= 12:
        score += 10

    # Skills Score
    skills = [s["name"].lower() for s in c["skills"]]

    for skill in skills:
        score += GOLD_SKILLS.get(skill, 0)

    # Career Score
    for job in c["career_history"]:

        desc = job["description"].lower()

        for kw in CAREER_KEYWORDS:
            if kw in desc:
                score += 5

    return score

In [71]:
import json

with open(PATH, "r", encoding="utf-8") as f:
    candidate = json.loads(next(f))

print(score_candidate(candidate))

20


In [72]:
top_candidates = []

with open(PATH, "r", encoding="utf-8") as f:

    for line in f:

        c = json.loads(line)

        score = score_candidate(c)

        top_candidates.append(
            (
                score,
                c["candidate_id"],
                c["profile"]["current_title"]
            )
        )

print(len(top_candidates))

100000


In [73]:
top100 = sorted(
    top_candidates,
    reverse=True
)[:100]

print(len(top100))

100


In [74]:
for rank, candidate in enumerate(top100[:10], start=1):
    print(rank, candidate)

1 (153, 'CAND_0018499', 'Senior Machine Learning Engineer')
2 (138, 'CAND_0081846', 'Lead AI Engineer')
3 (131, 'CAND_0055905', 'Senior Machine Learning Engineer')
4 (119, 'CAND_0077337', 'Staff Machine Learning Engineer')
5 (114, 'CAND_0094759', 'Lead AI Engineer')
6 (108, 'CAND_0071974', 'Senior AI Engineer')
7 (108, 'CAND_0044855', 'Senior Data Scientist')
8 (105, 'CAND_0041611', 'Staff Machine Learning Engineer')
9 (103, 'CAND_0070398', 'Machine Learning Engineer')
10 (102, 'CAND_0092278', 'Senior NLP Engineer')


In [75]:
import pandas as pd

rows = []

for rank, item in enumerate(top100, start=1):

    score, candidate_id, title = item

    rows.append({
        "candidate_id": candidate_id,
        "rank": rank,
        "score": score,
        "reasoning": f"{title} selected based on skills, experience and career relevance."
    })

submission = pd.DataFrame(rows)

print(submission.head())

   candidate_id  rank  score  \
0  CAND_0018499     1    153   
1  CAND_0081846     2    138   
2  CAND_0055905     3    131   
3  CAND_0077337     4    119   
4  CAND_0094759     5    114   

                                           reasoning  
0  Senior Machine Learning Engineer selected base...  
1  Lead AI Engineer selected based on skills, exp...  
2  Senior Machine Learning Engineer selected base...  
3  Staff Machine Learning Engineer selected based...  
4  Lead AI Engineer selected based on skills, exp...  


In [76]:
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("CSV Saved Successfully")

CSV Saved Successfully


In [77]:
# Behavioral Signals

signals = c["redrob_signals"]

score += signals.get("github_activity_score", 0) * 0.25

score += signals.get("profile_completeness_score", 0) * 0.08

score += signals.get("interview_completion_rate", 0) * 8

score += signals.get("recruiter_response_rate", 0) * 8

score += signals.get("saved_by_recruiters_30d", 0) * 0.4

# Open To Work Bonus

if signals.get("open_to_work_flag", False):
    score += 10

# Penalties

if not signals.get("open_to_work_flag", False):
    score -= 10

if signals.get("recruiter_response_rate", 0) < 0.20:
    score -= 10

if signals.get("avg_response_time_hours", 999) > 168:
    score -= 8

if signals.get("profile_completeness_score", 0) < 60:
    score -= 5

In [78]:
def score_candidate(c):

    score = 0

    # Experience Score
    ...

    # Skills Score
    ...

    # Career Score
    ...

    # Behavioral Signals
    signals = c["redrob_signals"]

    score += signals.get("github_activity_score", 0) * 0.25
    score += signals.get("profile_completeness_score", 0) * 0.08
    score += signals.get("interview_completion_rate", 0) * 8
    score += signals.get("recruiter_response_rate", 0) * 8
    score += signals.get("saved_by_recruiters_30d", 0) * 0.4

    if signals.get("open_to_work_flag", False):
        score += 10

    if not signals.get("open_to_work_flag", False):
        score -= 10

    if signals.get("recruiter_response_rate", 0) < 0.20:
        score -= 10

    if signals.get("avg_response_time_hours", 999) > 168:
        score -= 8

    if signals.get("profile_completeness_score", 0) < 60:
        score -= 5

    return round(score, 2)

In [79]:
top_candidates = []

with open(PATH, "r", encoding="utf-8") as f:

    for line in f:

        c = json.loads(line)

        score = score_candidate(c)

        top_candidates.append(
            (
                score,
                c["candidate_id"],
                c["profile"]["current_title"]
            )
        )

print("Total Candidates =", len(top_candidates))

Total Candidates = 100000


In [80]:
top100 = sorted(
    top_candidates,
    reverse=True
)[:100]

print("Top 100 =", len(top100))

Top 100 = 100


In [81]:
for rank, candidate in enumerate(top100[:10], start=1):
    print(rank, candidate)

1 (75.61, 'CAND_0076163', 'NLP Engineer')
2 (74.15, 'CAND_0061257', 'Staff Machine Learning Engineer')
3 (72.79, 'CAND_0020708', 'Search Engineer')
4 (72.76, 'CAND_0093912', 'Senior Data Scientist')
5 (71.02, 'CAND_0019480', 'NLP Engineer')
6 (70.63, 'CAND_0051615', 'Search Engineer')
7 (70.55, 'CAND_0071974', 'Senior AI Engineer')
8 (70.44, 'CAND_0077285', 'Recommendation Systems Engineer')
9 (69.46, 'CAND_0039754', 'Senior Applied Scientist')
10 (69.02, 'CAND_0075481', 'AI Research Engineer')


In [82]:
import pandas as pd

rows = []

for rank, item in enumerate(top100, start=1):

    score, candidate_id, title = item

    rows.append({
        "candidate_id": candidate_id,
        "rank": rank,
        "score": score,
        "reasoning": f"{title} selected based on experience, skills, career history and behavioral signals."
    })

submission = pd.DataFrame(rows)

print(submission.head())

   candidate_id  rank  score  \
0  CAND_0076163     1  75.61   
1  CAND_0061257     2  74.15   
2  CAND_0020708     3  72.79   
3  CAND_0093912     4  72.76   
4  CAND_0019480     5  71.02   

                                           reasoning  
0  NLP Engineer selected based on experience, ski...  
1  Staff Machine Learning Engineer selected based...  
2  Search Engineer selected based on experience, ...  
3  Senior Data Scientist selected based on experi...  
4  NLP Engineer selected based on experience, ski...  


In [83]:
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("Submission saved successfully")

Submission saved successfully


In [84]:
print("Total Submission Rows =", len(submission))
submission.head(10)

Total Submission Rows = 100


,candidate_id,rank,score,reasoning
0,CAND_0076163,1,75.61,"NLP Engineer selected based on experience, ski..."
1,CAND_0061257,2,74.15,Staff Machine Learning Engineer selected based...
2,CAND_0020708,3,72.79,"Search Engineer selected based on experience, ..."
3,CAND_0093912,4,72.76,Senior Data Scientist selected based on experi...
4,CAND_0019480,5,71.02,"NLP Engineer selected based on experience, ski..."
5,CAND_0051615,6,70.63,"Search Engineer selected based on experience, ..."
6,CAND_0071974,7,70.55,Senior AI Engineer selected based on experienc...
7,CAND_0077285,8,70.44,Recommendation Systems Engineer selected based...
8,CAND_0039754,9,69.46,Senior Applied Scientist selected based on exp...
9,CAND_0075481,10,69.02,AI Research Engineer selected based on experie...


In [85]:
print("Unique Ranks:", submission["rank"].nunique())
print("Min Rank:", submission["rank"].min())
print("Max Rank:", submission["rank"].max())

Unique Ranks: 100
Min Rank: 1
Max Rank: 100


In [86]:
print(
    "Duplicate Candidate IDs:",
    submission["candidate_id"].duplicated().sum()
)

Duplicate Candidate IDs: 0


In [87]:
submission.sample(5)

,candidate_id,rank,score,reasoning
43,CAND_0065633,44,62.87,"Data Scientist selected based on experience, s..."
46,CAND_0042506,47,62.66,"Search Engineer selected based on experience, ..."
41,CAND_0097079,42,62.98,Computer Vision Engineer selected based on exp...
11,CAND_0080465,12,68.49,"ML Engineer selected based on experience, skil..."
77,CAND_0048375,78,59.73,Computer Vision Engineer selected based on exp...


In [88]:
print(submission.columns.tolist())
print(len(submission))
print(submission["candidate_id"].duplicated().sum())
print(submission["rank"].nunique())

['candidate_id', 'rank', 'score', 'reasoning']
100
0
100
